### Seismic SPH and Ice-Water Collisions

#### Smoothed Particle Hydrodynamical Simulation

Run simulation for the experimental benchmark (standard resolution)

In [ ]:
import hyglac as ss
import numpy as np
import pandas as pd
import time, os

import matplotlib.colors as colors
from matplotlib import pyplot as plt
from matplotlib import cm, rc
from matplotlib.ticker import FormatStrFormatter, NullFormatter, LogLocator
from matplotlib.colors import BoundaryNorm
from scipy.signal import welch

Set up geometry of tank and initial conditions for fluid.  

In [ ]:
# set name of simulation
filename = 'scurve_lobovsky'

# set dimensions of simulation tank
x = 5.37 #5.366 # length in metres
y = 0.2 # width in metres
z = 1.5 # height in metres
t = 3 # duration in seconds
dt_out = 0.001 # duration in seconds

# resolution of simulation
dx = 1/100.0
dt = 5e-5

# define functions to set locations of fluid (initially) and obstacle particles
def set_fluid(x, y, z, tol):
    xtol, ytol, ztol = tol, tol, 0.01*tol
    # set region for fluid particles
    return np.where((0.0 + xtol < x) & (x <= 2.0 + xtol) & (-0.1 + ytol < y) & (y < 0.1 - ytol) \
                    & (0.0 + ztol < z) & (z <= 1.0 + ztol))[0]

def set_obstacle(x, y, z, tol):
    xtol, ytol, ztol = tol, tol, 0.01*tol
    # set region for obstacle particles
    return np.where((x <= 0.0 + xtol) | (0.1 - ytol <= y) | (y <= -0.1 + ytol) \
                    | (x >= 5.37 + xtol) | (z <= 0.0 + ztol))[0]

Run smoothed particle hydrodynamical simulation.

In [ ]:
ss.run_seismic_sph(x, y, z, t, dx, dt, set_fluid, set_obstacle, output_times=np.arange(0, t + dt_out/2, dt_out), \
                   filename='{}_time'.format(filename))

Plot output of simulation at given time step.

In [ ]:
ss.plot_seismic_sph(x, y, z, 10000, particle_type='fluid', filename=filename)

#### Extraction of focres by dynamic pressure smoothing

Find properties of collisions with tank wall.

In [ ]:
ss.seismic_sph_interactions(filename='{}_time'.format(filename), delta=0.01, dx=0.04, dy=0.04, dz=0.04, xrange=[4,6])

Generate force vectors from collision database.

In [ ]:
x0, y0, z0 = 5.37, 0., 0. # define location of point source; in metres
forces_df = ss.seismic_sph_forces_defaults(x0, y0, z0, filename='{}_time'.format(filename))
forces_df.to_csv('{}_forces.csv'.format(filename))